# DBSCAN: Density-Based Clustering

DBSCAN finds clusters of arbitrary shape and automatically identifies noise points.

1. **Algorithm** - Core points, border points, noise
2. **Parameter Tuning** - eps (neighborhood radius) and min_samples
3. **Comparison with K-Means** on non-spherical data

**Dataset**: Synthetic moons and circles

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_moons, make_circles, make_blobs
from sklearn.cluster import DBSCAN, KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

sns.set_theme(style="whitegrid")

In [ ]:
# Generate challenging datasets
datasets = {
    "Moons": make_moons(n_samples=500, noise=0.08, random_state=42),
    "Circles": make_circles(n_samples=500, noise=0.05, factor=0.5, random_state=42),
    "Blobs with noise": (lambda: (
        np.vstack([make_blobs(n_samples=400, centers=3, cluster_std=0.5, random_state=42)[0],
                   np.random.uniform(-10, 10, (50, 2))]),
        np.hstack([make_blobs(n_samples=400, centers=3, cluster_std=0.5, random_state=42)[1],
                   np.full(50, -1)])
    ))(),
}

fig, axes = plt.subplots(len(datasets), 3, figsize=(18, 5 * len(datasets)))

for row, (name, (X, y_true)) in enumerate(datasets.items()):
    X = StandardScaler().fit_transform(X)
    
    # True labels
    axes[row, 0].scatter(X[:, 0], X[:, 1], c=y_true, cmap="viridis", s=15)
    axes[row, 0].set_title(f"{name}: True Labels")
    
    # K-Means
    n_clusters = len(set(y_true)) - (1 if -1 in y_true else 0)
    km_labels = KMeans(n_clusters=max(n_clusters, 2), n_init=10, random_state=42).fit_predict(X)
    axes[row, 1].scatter(X[:, 0], X[:, 1], c=km_labels, cmap="viridis", s=15)
    axes[row, 1].set_title(f"K-Means (K={max(n_clusters, 2)})")
    
    # DBSCAN
    db = DBSCAN(eps=0.3, min_samples=10)
    db_labels = db.fit_predict(X)
    n_noise = (db_labels == -1).sum()
    axes[row, 2].scatter(X[:, 0], X[:, 1], c=db_labels, cmap="viridis", s=15)
    axes[row, 2].set_title(f"DBSCAN (clusters={len(set(db_labels)) - (1 if -1 in db_labels else 0)}, noise={n_noise})")

plt.tight_layout()
plt.show()

## Choosing eps: K-Distance Plot

In [ ]:
X_moons = StandardScaler().fit_transform(make_moons(n_samples=500, noise=0.08, random_state=42)[0])

# K-distance plot: sort distances to k-th nearest neighbor
k = 10
nn = NearestNeighbors(n_neighbors=k)
nn.fit(X_moons)
distances, _ = nn.kneighbors(X_moons)
k_distances = np.sort(distances[:, -1])

plt.figure(figsize=(8, 5))
plt.plot(k_distances, color="teal")
plt.xlabel("Points (sorted)")
plt.ylabel(f"{k}-th Nearest Neighbor Distance")
plt.title("K-Distance Plot (look for the 'elbow')")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Key Takeaways

1. **DBSCAN finds arbitrary-shaped clusters** - unlike K-Means
2. **Automatically detects noise** - points labeled as -1
3. **No need to specify K** - number of clusters is discovered
4. **eps and min_samples control granularity** - use K-distance plot to set eps
5. **Struggles with varying-density clusters** - consider HDBSCAN for that